In [2]:
# ============================================================
# CELL 1 — CONNECT TO POSTGRESQL
# ============================================================

import pandas as pd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
import time

# PostgreSQL connection
# quote_plus handles special characters in password like @, #, $ etc.
DB_USER     = 'postgres'
DB_PASSWORD = quote_plus('NewStrongPassword@123')  # Replace yourpassword with your actual password
DB_HOST     = 'localhost'
DB_PORT     = '5432'
DB_NAME     = 'franchise_intelligence'

engine = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)

# Test connection
with engine.connect() as conn:
    result = conn.execute(text('SELECT version()'))
    print("✅ Connected to PostgreSQL successfully")
    print(f"Version: {result.fetchone()[0]}")

✅ Connected to PostgreSQL successfully
Version: PostgreSQL 17.6 on x86_64-windows, compiled by msvc-19.44.35213, 64-bit


In [3]:
# ============================================================
# CELL 2 — LOAD ALL CSV FILES INTO POSTGRESQL
# ============================================================

import os

raw_path = r'D:\Projects\End-to-end projects\12. Franchise Intelligence System\Data\Raw'

# Load CSVs back into dataframes
print("📂 Reading CSV files...")
outlet_master       = pd.read_csv(f'{raw_path}\\outlet_master.csv')
daily_transactions  = pd.read_csv(f'{raw_path}\\daily_transactions.csv')
monthly_perf        = pd.read_csv(f'{raw_path}\\monthly_performance.csv')
staff_df            = pd.read_csv(f'{raw_path}\\staff_records.csv')
reviews_df          = pd.read_csv(f'{raw_path}\\customer_reviews.csv')
print("✅ All CSV files loaded into memory\n")

# ── Load outlet_master ──────────────────────────────────────
print("⏳ Loading outlet_master...")
start = time.time()
outlet_master.to_sql('outlet_master', engine, 
                      if_exists='append', index=False)
print(f"✅ outlet_master loaded — {len(outlet_master):,} rows in {time.time()-start:.1f}s")

# ── Load daily_transactions ─────────────────────────────────
print("\n⏳ Loading daily_transactions (largest table)...")
start = time.time()
daily_transactions.to_sql('daily_transactions', engine,
                           if_exists='append', index=False,
                           chunksize=10000, method='multi')
print(f"✅ daily_transactions loaded — {len(daily_transactions):,} rows in {time.time()-start:.1f}s")

# ── Load monthly_performance ────────────────────────────────
print("\n⏳ Loading monthly_performance...")
start = time.time()
monthly_perf.to_sql('monthly_performance', engine,
                     if_exists='append', index=False)
print(f"✅ monthly_performance loaded — {len(monthly_perf):,} rows in {time.time()-start:.1f}s")

# ── Load staff_records ──────────────────────────────────────
print("\n⏳ Loading staff_records...")
start = time.time()
staff_df.to_sql('staff_records', engine,
                 if_exists='append', index=False)
print(f"✅ staff_records loaded — {len(staff_df):,} rows in {time.time()-start:.1f}s")

# ── Load customer_reviews ───────────────────────────────────
print("\n⏳ Loading customer_reviews...")
start = time.time()
reviews_df.to_sql('customer_reviews', engine,
                   if_exists='append', index=False)
print(f"✅ customer_reviews loaded — {len(reviews_df):,} rows in {time.time()-start:.1f}s")

print("\n" + "="*55)
print("✅ ALL TABLES LOADED INTO POSTGRESQL SUCCESSFULLY")
print("="*55)

📂 Reading CSV files...
✅ All CSV files loaded into memory

⏳ Loading outlet_master...
✅ outlet_master loaded — 85 rows in 0.1s

⏳ Loading daily_transactions (largest table)...
✅ daily_transactions loaded — 681,949 rows in 532.0s

⏳ Loading monthly_performance...
✅ monthly_performance loaded — 1,530 rows in 0.4s

⏳ Loading staff_records...
✅ staff_records loaded — 1,530 rows in 0.1s

⏳ Loading customer_reviews...
✅ customer_reviews loaded — 28,585 rows in 2.3s

✅ ALL TABLES LOADED INTO POSTGRESQL SUCCESSFULLY


In [4]:
# ============================================================
# CELL 3 — VERIFY ALL TABLES IN POSTGRESQL
# ============================================================

verification_queries = {
    'outlet_master':       'SELECT COUNT(*) FROM outlet_master',
    'daily_transactions':  'SELECT COUNT(*) FROM daily_transactions',
    'monthly_performance': 'SELECT COUNT(*) FROM monthly_performance',
    'staff_records':       'SELECT COUNT(*) FROM staff_records',
    'customer_reviews':    'SELECT COUNT(*) FROM customer_reviews',
}

print("📊 POSTGRESQL TABLE VERIFICATION")
print("="*45)

total = 0
with engine.connect() as conn:
    for table, query in verification_queries.items():
        count = conn.execute(text(query)).fetchone()[0]
        total += count
        print(f"  {table:<25} {count:>10,} rows")

print("="*45)
print(f"  {'TOTAL':<25} {total:>10,} rows")
print("\n✅ All tables verified in PostgreSQL")

📊 POSTGRESQL TABLE VERIFICATION
  outlet_master                     85 rows
  daily_transactions           681,949 rows
  monthly_performance            1,530 rows
  staff_records                  1,530 rows
  customer_reviews              28,585 rows
  TOTAL                        713,679 rows

✅ All tables verified in PostgreSQL
